<a href="https://colab.research.google.com/github/parksunnysky05/feature-story/blob/main/MotorPH_Dataset_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# LOAD THE DATASETS
products = pd.read_csv('/content/MotorPH_Products_List_2025.csv')
sales = pd.read_csv('/content/MotorPH_Sales Data-3rd Quarter-Year 2025.csv')

print("PRODUCTS DATASET:")
display(products.head())

print("\nSALES DATASET:")
display(sales.head())

PRODUCTS DATASET:


,EntrNo,EntrName,EntrDetails,Manufacturing Date,Acquisiton,UnitPrice
0,1,Honda CBR500R,"Sport / Parallel-twin, liquid-cooled, 471cc, 6...",2021,2023,389000
1,2,Yamaha MT-15,"Naked bike / Single cylinder, liquid-cooled, 1...",2022,2023,178000
2,3,Kawasaki Ninja 400,"Sport / Parallel-twin, liquid-cooled, 399cc, 6...",2023,2024,340000
3,4,Suzuki Raider R150 Fi,"Underbone / Single cylinder, liquid-cooled, 15...",2020,2021,119900
4,5,Royal Enfield Classic 350,"Classic / Single cylinder, air-cooled, 349cc, ...",2022,2023,250000



SALES DATASET:


,date,client_type,product,unitprice,quantity,total,payment
0,7/22/2025,Retail,Zontes GK350,269000,20,5380000,Cash
1,6/15/2025,Wholesale,Yamaha FZi 150,99900,18,1798200,Credit card
2,8/11/2025,Wholesale,Zontes GK350,269000,41,11029000,Credit card
3,7/31/2025,Retail,Italjet Dragster 200,867000,50,14450000,Cash
4,6/21/2025,Wholesale,Zontes 310X,248000,7,1736000,Cash


In [ ]:
# REUSABLE HELPER FUNCTIONS

def inspect_dataset(df, name):
    """Print structure, missing values, and duplicates for any dataset."""
    print(f"===== {name} INFO =====")
    df.info()
    print(f"\n{name} missing values:\n{df.isnull().sum()}")
    print(f"\n{name} duplicate rows: {df.duplicated().sum()}")
    print(f"\n{name} columns: {df.columns.tolist()}\n")


def clean_text_column(series):
    """Standardize a text column: trim whitespace, Title Case."""
    return series.str.strip().str.title()


def validate_dataset(df, name):
    """Re-check missing values and duplicates after cleaning."""
    print(f"===== {name} FINAL CHECK =====")
    print(f"Missing values:\n{df.isnull().sum()}")
    print(f"Duplicate rows: {df.duplicated().sum()}\n")


inspect_dataset(products, "PRODUCTS DATASET")
inspect_dataset(sales, "SALES DATASET")

===== PRODUCTS DATASET INFO =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   EntrNo              50 non-null     int64 
 1   EntrName            50 non-null     object
 2   EntrDetails         50 non-null     object
 3   Manufacturing Date  50 non-null     int64 
 4   Acquisiton          50 non-null     int64 
 5   UnitPrice           50 non-null     int64 
dtypes: int64(4), object(2)
memory usage: 2.5+ KB

PRODUCTS DATASET missing values:
EntrNo                0
EntrName              0
EntrDetails           0
Manufacturing Date    0
Acquisiton            0
UnitPrice             0
dtype: int64

PRODUCTS DATASET duplicate rows: 0

PRODUCTS DATASET columns: ['EntrNo', 'EntrName', 'EntrDetails', 'Manufacturing Date', 'Acquisiton', 'UnitPrice']

===== SALES DATASET INFO =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 en

In [ ]:
# DATA PREPROCESSING - PRODUCTS DATASET

def clean_products(df):
    """Transform the raw products dataset into the required report format."""
    clean = df.copy()

    clean = clean.rename(columns={
        'EntrNo': 'Product ID Number',
        'EntrName': 'Product Name',
        'UnitPrice': 'Unit Price',
        'Manufacturing Date': 'Date of Manufacturing',
        'Acquisiton': 'Date of Acquisition'
    })

    # Product Type is the text before "/" in EntrDetails
    clean['Product Type'] = clean['EntrDetails'].str.split('/').str[0].str.strip()
    clean = clean.drop(columns=['EntrDetails'])

    clean = clean[[
        'Product ID Number', 'Product Name', 'Product Type',
        'Unit Price', 'Date of Manufacturing', 'Date of Acquisition'
    ]]

    clean['Product Name'] = clean['Product Name'].str.strip()
    clean['Product Type'] = clean['Product Type'].str.strip()
    clean['Product ID Number'] = range(1, len(clean) + 1)  # guarantee sequential IDs

    return clean.drop_duplicates()


products_clean = clean_products(products)

print("===== CLEANED PRODUCTS DATASET =====")
display(products_clean.head(10))

===== CLEANED PRODUCTS DATASET =====


,Product ID Number,Product Name,Product Type,Unit Price,Date of Manufacturing,Date of Acquisition
0,1,Honda CBR500R,Sport,389000,2021,2023
1,2,Yamaha MT-15,Naked bike,178000,2022,2023
2,3,Kawasaki Ninja 400,Sport,340000,2023,2024
3,4,Suzuki Raider R150 Fi,Underbone,119900,2020,2021
4,5,Royal Enfield Classic 350,Classic,250000,2022,2023
5,6,CFMoto 300NK,Naked bike,165000,2021,2022
6,7,Bristol Veloce 500,Cafe racer,328000,2022,2023
7,8,KTM Duke 200,Naked bike,178000,2023,2024
8,9,Yamaha XSR155,Retro,182000,2022,2023
9,10,Bajaj Dominar 400,Sport touring,199900,2021,2022


In [ ]:
# DATA PREPROCESSING - SALES DATASET

def fix_corrupted_product_names(sales_df, catalog_names):
    """
    Some product names in the sales file have their last character
    replaced with 'x' (e.g. 'Honda ADV 16x' -> 'Honda ADV 160').
    Match on every character except the last to recover the real name.
    """
    stem_to_name = {name[:-1]: name for name in catalog_names}

    def fix_name(name):
        if name in catalog_names:
            return name
        if isinstance(name, str) and name.endswith('x') and name[:-1] in stem_to_name:
            return stem_to_name[name[:-1]]
        return name

    sales_df['product'] = sales_df['product'].apply(fix_name)
    return sales_df


def fix_unit_prices(sales_df, price_map):
    """
    Some rows have a unit price that doesn't match the catalog
    (found to be inflated ~3x on 19 rows). Recompute both unitprice
    and total from the trusted catalog price.
    """
    sales_df['unitprice'] = sales_df['product'].map(price_map).fillna(sales_df['unitprice'])
    sales_df['total'] = sales_df['unitprice'] * sales_df['quantity']
    return sales_df


def clean_sales(df, products_clean):
    """Full cleaning pipeline for the sales dataset."""
    clean = df.copy()

    # Fix product-name corruption and price errors by cross-referencing the catalog
    catalog_names = products_clean['Product Name'].tolist()
    price_map = products_clean.set_index('Product Name')['Unit Price'].to_dict()
    clean = fix_corrupted_product_names(clean, catalog_names)
    clean = fix_unit_prices(clean, price_map)

    # Handle missing values
    clean = clean.dropna(subset=['date'])
    clean['client_type'] = clean['client_type'].fillna('Unknown')
    clean['payment'] = clean['payment'].fillna('Unknown')

    # Standardize text columns using the shared helper function
    clean['client_type'] = clean_text_column(clean['client_type'])
    clean['payment'] = clean_text_column(clean['payment'])
    clean['product'] = clean['product'].str.strip()

    # Convert and validate dates; drop rows that are truly unparseable
    clean['date'] = pd.to_datetime(clean['date'], errors='coerce')
    clean = clean.dropna(subset=['date'])

    return clean.drop_duplicates().reset_index(drop=True)


sales_clean = clean_sales(sales, products_clean)

print("===== CLEANED SALES DATASET =====")
display(sales_clean.head())

unmatched = set(sales_clean['product']) - set(products_clean['Product Name'])
print("\nProduct names still unmatched to catalog:", unmatched)

===== CLEANED SALES DATASET =====


,date,client_type,product,unitprice,quantity,total,payment
0,2025-07-22,Retail,Zontes GK350,269000,20,5380000,Cash
1,2025-06-15,Wholesale,Yamaha FZi 150,99900,18,1798200,Credit Card
2,2025-08-11,Wholesale,Zontes GK350,269000,41,11029000,Credit Card
3,2025-07-31,Retail,Italjet Dragster 200,289000,50,14450000,Cash
4,2025-06-21,Wholesale,Zontes 310X,248000,7,1736000,Cash



Product names still unmatched to catalog: set()


In [ ]:
# FINAL VALIDATION

validate_dataset(products_clean, "PRODUCTS")
validate_dataset(sales_clean, "SALES")

# Confirm total = unitprice x quantity holds for every row
incorrect_totals = sales_clean[
    sales_clean['total'] != sales_clean['unitprice'] * sales_clean['quantity']
]
print("Incorrect total calculations remaining:", len(incorrect_totals))

===== PRODUCTS FINAL CHECK =====
Missing values:
Product ID Number        0
Product Name             0
Product Type             0
Unit Price               0
Date of Manufacturing    0
Date of Acquisition      0
dtype: int64
Duplicate rows: 0

===== SALES FINAL CHECK =====
Missing values:
date           0
client_type    0
product        0
unitprice      0
quantity       0
total          0
payment        0
dtype: int64
Duplicate rows: 0

Incorrect total calculations remaining: 0


In [ ]:
# EXPORT CLEANED DATASETS

products_clean.to_csv('MotorPH_Processed_Products.csv', index=False)
sales_clean.to_csv('MotorPH_Processed_Sales.csv', index=False)

print("Files exported successfully!")

Files exported successfully!
